## ***Initials***

In [41]:
import pandas as pd
from geopy.distance import geodesic
from shapely.geometry import Point
import geopandas as gpd
from shapely.ops import nearest_points

## ***Load the Data***

In [24]:
df = pd.read_csv('C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\5. Neighborhoods\\Extracted Neighborhoods\\neighborhoods_cleaned.csv')
df.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...


## ***Insert: Distance from Center***

In [25]:
volos_center = (39.3666, 22.9507)

In [26]:
# Compute distance to Volos center
df['distance_to_volos_center_km'] = df.apply(
    lambda row: geodesic((row['Centroid_y'], row['Centroid_x']), volos_center).km,
    axis=1
)

In [27]:
df.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry,distance_to_volos_center_km
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166


## ***Insert: Distance from Port***

In [28]:
volos_port = (39.35887819842874, 22.944538825211723)

In [29]:
# Compute distance from each grid centroid to Volos port
df['distance_to_volos_port_km'] = df.apply(
    lambda row: geodesic((row['Centroid_y'], row['Centroid_x']), volos_port).km,
    axis=1
)

In [30]:
df.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry,distance_to_volos_center_km,distance_to_volos_port_km
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015,5.418577
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475,11.015584
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666,9.191001
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184,1.978856
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166,2.529372


## ***Insert: Distance from Highway***

In [31]:
# Convert to GeoDataFrame using centroids
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Centroid_x'], df['Centroid_y']),
    crs="EPSG:4326"  # assuming lon/lat
)

In [32]:
# Load the main road shapefile
main_roads = gpd.read_file("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\4. Gridding\\shapefiles - main roads\\main_roads_magnesia.shp").to_crs("EPSG:4326")

In [33]:
# Merge all line features into one geometry
road_union = main_roads.unary_union

C:\Users\Giorgos\AppData\Local\Temp\ipykernel_27916\3614782751.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  road_union = main_roads.unary_union


In [34]:
# Calculate geodesic distance (in km) from each centroid to the nearest point on the road network
def compute_distance_km(point):
    nearest_point = road_union.interpolate(road_union.project(point))
    return geodesic((point.y, point.x), (nearest_point.y, nearest_point.x)).km

In [35]:
# Apply to each neighborhood centroid
gdf['dist_to_main_road_km'] = gdf['geometry'].apply(compute_distance_km)

In [36]:
gdf.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry,distance_to_volos_center_km,distance_to_volos_port_km,geometry,dist_to_main_road_km
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015,5.418577,POINT (22.88294 39.34911),0.443434
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475,11.015584,POINT (22.8369 39.30543),2.299921
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666,9.191001,POINT (22.83817 39.35305),0.482308
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184,1.978856,POINT (22.92406 39.36694),0.058142
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166,2.529372,POINT (22.92266 39.3437),0.224486


## ***Insert: Distance from Transport Station***

In [37]:
bus_stops = gpd.read_file("C:/Users/Giorgos/Desktop/HMMY/10ο Εξάμηνο/Διπλωματική/4. Gridding/shapefiles - transport stops/transport-clean.gpkg")

In [38]:
# Combine all bus stops into a single geometry collection
bus_union = bus_stops.unary_union

C:\Users\Giorgos\AppData\Local\Temp\ipykernel_27916\1027943568.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  bus_union = bus_stops.unary_union


In [39]:
# Function to compute geodesic distance in km from a grid centroid to the nearest bus stop
def compute_distance_km(point):
    nearest_stop = nearest_points(point, bus_union)[1]
    return geodesic((point.y, point.x), (nearest_stop.y, nearest_stop.x)).km

In [42]:
# Apply the function to each grid centroid
gdf['dist_to_bus_stop_km'] = gdf['geometry'].apply(compute_distance_km)

In [43]:
gdf.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry,distance_to_volos_center_km,distance_to_volos_port_km,geometry,dist_to_main_road_km,dist_to_bus_stop_km
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015,5.418577,POINT (22.88294 39.34911),0.443434,1.569912
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475,11.015584,POINT (22.8369 39.30543),2.299921,2.410375
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666,9.191001,POINT (22.83817 39.35305),0.482308,4.965561
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184,1.978856,POINT (22.92406 39.36694),0.058142,0.063879
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166,2.529372,POINT (22.92266 39.3437),0.224486,0.246542


## ***Insert: Distance from University***

In [44]:
universities = gpd.read_file("C:/Users/Giorgos/Desktop/HMMY/10ο Εξάμηνο/Διπλωματική/4. Gridding/shapefiles - university/universities-magnesia.gpkg")

In [45]:
# Merge all university points into one geometry collection
uni_union = universities.unary_union

C:\Users\Giorgos\AppData\Local\Temp\ipykernel_27916\2462736085.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  uni_union = universities.unary_union


In [46]:
# Define function to compute geodesic distance
def dist_to_university(point):
    nearest = nearest_points(point, uni_union)[1]
    return geodesic((point.y, point.x), (nearest.y, nearest.x)).km

In [47]:
gdf['dist_to_university_km'] = gdf['geometry'].apply(dist_to_university)

In [48]:
gdf.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry,distance_to_volos_center_km,distance_to_volos_port_km,geometry,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015,5.418577,POINT (22.88294 39.34911),0.443434,1.569912,4.273031
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475,11.015584,POINT (22.8369 39.30543),2.299921,2.410375,10.100771
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666,9.191001,POINT (22.83817 39.35305),0.482308,4.965561,7.989610
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184,1.978856,POINT (22.92406 39.36694),0.058142,0.063879,0.866021
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166,2.529372,POINT (22.92266 39.3437),0.224486,0.246542,1.913456


## ***Save the File***

In [54]:
gdf.drop(columns=['geometry'], inplace=True)

In [55]:
gdf.head()

,Municipal Community,Neighborhood,Centroid_x,Centroid_y,Area_km2,Geometry,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,Δημοτική Κοινότητα Διμηνίου,Δημίνι,22.882945,39.349112,37.344776,MULTIPOLYGON (((406692.718139543 4357521.85294...,6.154015,5.418577,0.443434,1.569912,4.273031
1,Δημοτική Κοινότητα Σέσκλου,Χρυσή Ακτή Παναγίας,22.836899,39.305430,9.983032,MULTIPOLYGON (((396744.165394512 4351195.65810...,11.932475,11.015584,2.299921,2.410375,10.100771
2,Δημοτική Κοινότητα Σέσκλου,Σέσκλο,22.838168,39.353051,27.367312,MULTIPOLYGON (((400839.361742302 4353031.79092...,9.814666,9.191001,0.482308,4.965561,7.989610
3,Δημοτική Κοινότητα Βόλου,Άγιοι Ανάργυροι,22.924059,39.366937,0.774436,MULTIPOLYGON (((406589.250339913 4357224.83344...,2.296184,1.978856,0.058142,0.063879,0.866021
4,Δημοτική Κοινότητα Βόλου,Αϊβαλιώτικα,22.922660,39.343696,4.848904,MULTIPOLYGON (((407619.510540358 4356306.42684...,3.508166,2.529372,0.224486,0.246542,1.913456


In [56]:
gdf.to_csv("neighborhoods_enriched.csv", index=False)